In [25]:
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error as mape
# reads csv files into dataframes
df0 = pd.read_csv("CRMLSSold202506.csv")
df1 = pd.read_csv("CRMLSSold202507.csv")
df2 = pd.read_csv("CRMLSSold202508.csv")
df3 = pd.read_csv("CRMLSSold202509.csv")
df4 = pd.read_csv("CRMLSSold202510.csv")
df5 = pd.read_csv("CRMLSSold202511.csv")
df6 = pd.read_csv("CRMLSSold202512.csv")
# Creates train test split
df_train = pd.concat([df0, df1, df2, df3, df4, df5], axis = 0)
df_test = df6
# Narrows down the data
df_train = df_train[(df_train["PropertyType"] == "Residential") & (df_train["PropertySubType"] == "SingleFamilyResidence")]
df_test = df_test[(df_test["PropertyType"] == "Residential") & (df_test["PropertySubType"] == "SingleFamilyResidence")]
# df_to_use.columns
# df_to_use.shape

# filters out invalid data
df_train = df_train[(df_train["ClosePrice"] > 0) & (df_train["LivingArea"] > 0)]
df_train = df_train[df_train["StateOrProvince"] == "CA"] 
df_train = df_train.dropna(subset = ["Latitude", "Longitude"])
orig_len = len(df_train)
df_train['CloseDate'] = pd.to_datetime(df_train['CloseDate'])

df_train = (
    df_train.sort_values('CloseDate', ascending=False)
      .drop_duplicates(subset='ListingKey', keep='first')
)

df_test = df_test[(df_test["ClosePrice"] > 0) & (df_test["LivingArea"] > 0)]
df_test = df_test[df_test["StateOrProvince"] == "CA"]
df_test = df_test.dropna(subset = ["Latitude", "Longitude"])
orig_test_len = len(df_test)

df_test = (df_test.sort_values('ClosePrice', ascending=False).drop_duplicates(subset='ListingKey', keep ='first'))

df_train["StateOrProvince"].value_counts()
df_train.columns
df_train["CloseDate"]
print(len(df_train), len(df_test), orig_len, orig_test_len)


C:\Users\jehil\AppData\Local\Temp\ipykernel_39784\1197617050.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df0 = pd.read_csv("CRMLSSold202506.csv")


68384 10445 68419 10445


In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree

def add_spatial_features(df, k=6):
    """
    Step 3: Feature Engineering (Spatial Data Science).
    
    Standard regression models fail to capture "Location".
    My solution: Create a 'Spatial Lag' feature.
    
    Logic: I use a BallTree (KNN) to find the nearest 5 neighbors for every house 
    and calculate their average price. This quantifies the 'neighborhood value'.
    """
    print(">> Engineering SpatialLag_Price (KNN algorithm)...")
    
    # Haversine metric requires radians
    df['lat_rad'] = np.radians(df['Latitude'])
    df['lon_rad'] = np.radians(df['Longitude'])
    
    # Build the tree for fast spatial indexing
    tree = BallTree(df[['lat_rad', 'lon_rad']].values, metric='haversine')
    
    # Query k nearest neighbors (1st one is the point itself, so we take k=6)
    dist, ind = tree.query(df[['lat_rad', 'lon_rad']].values, k=k)
    
    # Calculate mean price of neighbors (excluding the house itself)
    prices = df['ClosePrice'].values
    neighbor_indices = ind[:, 1:] 
    
    neighbor_prices = prices[neighbor_indices]
    df['SpatialLag_Price'] = np.mean(neighbor_prices, axis=1)
    
    return df

In [27]:
# add feature engineering to both train and test
df_train = add_spatial_features(df_train)
df_test = add_spatial_features(df_test)

>> Engineering SpatialLag_Price (KNN algorithm)...
>> Engineering SpatialLag_Price (KNN algorithm)...


In [28]:
df_train

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,lat_rad,lon_rad,SpatialLag_Price
8649,Downey,Downey,Wood,True,NaN,NaN,False,1198888.0,1136779157,Joshua@theave.agency,...,False,2.0,Orange Unified,92807,0.0,6000.0,NaN,0.590929,-2.055632,1084600.0
0,OrangeCounty,OrangeCounty,"Carpet,Tile",False,NaN,NaN,False,1250000.0,1147233684,mattkanoudi@gmail.com,...,False,2.0,Huntington Beach Union High,92646,0.0,5913.0,NaN,0.587758,-2.059405,1894036.0
13971,TriCounties,TriCounties,NaN,False,NaN,NaN,False,1298000.0,1126720375,elogisys@gmail.com,...,False,2.0,Glendora Unified,91741,0.0,12205.0,NaN,0.595780,-2.056464,1129400.0
3309,PacificWest,PacificWest,"Tile,Wood",False,NaN,NaN,False,769900.0,1140280497,Jamesj@sevengables.com,...,False,2.0,Ontario-Montclair,91786,0.0,10200.0,NaN,0.595003,-2.053225,716800.0
1740,Arcadia,Arcadia,"Carpet,Tile,Wood",False,NaN,NaN,False,1475000.0,1144787973,Garylorenzini@gmail.com,...,False,2.0,Arcadia Unified,91006,0.0,9329.0,NaN,0.595492,-2.059945,2132400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12230,Southland,Southland,"Concrete,Tile",True,NaN,NaN,True,1995000.0,1112047622,Gina.Michelle@TheAgencyRE.com,...,False,2.0,Los Angeles Unified,91367,0.0,11831.0,NaN,0.596359,-2.070754,1425000.0
19632,TriCounties,TriCounties,NaN,False,NaN,NaN,False,385000.0,1107686198,adnike@earthlink.net,...,False,2.0,Palmdale,93550,0.0,6500.0,NaN,0.603609,-2.061090,436200.0
10100,CitrusValley,CitrusValley,NaN,True,NaN,NaN,False,549900.0,1112350013,seanbrunsk@aol.com,...,False,0.0,Moreno Valley Unified,92553,0.0,7841.0,NaN,0.592274,-2.046127,511200.0
7778,PacificSouthwest,PacificSouthwest,NaN,False,NaN,NaN,True,425000.0,1112923920,henry@houseINsandiego.com,...,False,2.0,Kern Union,93308,0.0,17860.0,NaN,0.618038,-2.078432,448600.0


In [29]:

cols = ["ClosePrice", "BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude", "SpatialLag_Price"]
df_filtered_train = df_train[cols]
df_test_filtered = df_test[cols]

In [30]:
#finds number of na values for each feature
print(df_filtered_train.isna().sum())

df_test_filtered.isna().sum()


ClosePrice                  0
BedroomsTotal               0
BathroomsTotalInteger      11
LivingArea                  0
LotSizeSquareFeet        1153
YearBuilt                  35
Latitude                    0
Longitude                   0
SpatialLag_Price            0
dtype: int64


ClosePrice                 0
BedroomsTotal              0
BathroomsTotalInteger      0
LivingArea                 0
LotSizeSquareFeet        156
YearBuilt                  6
Latitude                   0
Longitude                  0
SpatialLag_Price           0
dtype: int64

In [31]:
#impute missing values within a range

# df_filtered_train = df_filtered_train.dropna(subset = ["YearBuilt", "BathroomsTotalInteger", "LotSizeSquareFeet"])


def random_sample_impute(train_df, test_df, column):
    observed_values = train_df[column].dropna().values
    
    
    train_missing = train_df[column].isna()
    train_df.loc[train_missing, column] = np.random.choice(
        observed_values,
        size=train_missing.sum(),
        replace=True
    )
    
    #
    test_missing = test_df[column].isna()
    test_df.loc[test_missing, column] = np.random.choice(
        observed_values,
        size=test_missing.sum(),
        replace=True
    )
    
    return train_df, test_df

df_filtered_train, df_test_filtered = random_sample_impute(df_filtered_train, df_test_filtered, "BathroomsTotalInteger")
df_filtered_train, df_test_filtered = random_sample_impute(df_filtered_train, df_test_filtered, "LotSizeSquareFeet")
df_filtered_train, df_test_filtered = random_sample_impute(df_filtered_train, df_test_filtered, "YearBuilt")


# df_test_filtered = df_test_filtered.dropna(subset = ["YearBuilt", "BathroomsTotalInteger", "LotSizeSquareFeet"])
# df_test_filtered

In [32]:

df_test_filtered.isna().sum()

ClosePrice               0
BedroomsTotal            0
BathroomsTotalInteger    0
LivingArea               0
LotSizeSquareFeet        0
YearBuilt                0
Latitude                 0
Longitude                0
SpatialLag_Price         0
dtype: int64

In [33]:
# df_filtered.isna().sum()
# 5108/df_filtered.shape[0]

In [34]:
# Encodes new construction variable into binary variable
# df_filtered_train["NewConstructionYN"][df_filtered_train["NewConstructionYN"] == False] = 0
# df_filtered_train["NewConstructionYN"][df_filtered_train["NewConstructionYN"] == True] = 1
# df_filtered_train["NewConstructionYN"].value_counts()

# df_test_filtered["NewConstructionYN"][df_test_filtered["NewConstructionYN"] == False] = 0
# df_test_filtered["NewConstructionYN"][df_test_filtered["NewConstructionYN"] == True] = 1
# df_test_filtered["NewConstructionYN"].value_counts()

In [35]:
#scales numerical variables only in training set
# from sklearn.preprocessing import StandardScaler
# df_filtered_train.dtypes
# num_cols = ["YearBuilt", "BathroomsTotalInteger", "BedroomsTotal", "LotSizeSquareFeet"]
# scaler = StandardScaler()
# df_filtered_train[num_cols] = scaler.fit_transform(df_filtered_train[num_cols])
# df_filtered_train
# df_test_filtered[num_cols] = scaler.transform(df_test_filtered[num_cols])

In [36]:
#remove bottom 0.5% and top 0.5% of ClosePrice values in the testing set
c_price = "ClosePrice"


In [37]:
#test set done separately
test_low = df_test_filtered[c_price].quantile(0.005)
test_high = df_test_filtered[c_price].quantile(0.995)
df_test_cleaned = df_test_filtered[(df_test_filtered[c_price] >= test_low) & (df_test_filtered[c_price] <= test_high)]
df_test_cleaned



,ClosePrice,BedroomsTotal,BathroomsTotalInteger,LivingArea,LotSizeSquareFeet,YearBuilt,Latitude,Longitude,SpatialLag_Price
11672,8500000.0,7.0,9.0,6210.0,10960.0,1928.0,34.071501,-118.336814,3715000.0
15375,8500000.0,4.0,8.0,5100.0,16167.0,1960.0,34.103839,-118.383941,6690000.0
20203,8350000.0,4.0,5.0,5764.0,19283.0,1939.0,34.103909,-118.383560,6347000.0
20061,8150000.0,4.0,6.0,4000.0,32578.0,1979.0,34.008534,-118.804572,6295000.0
12361,8100000.0,4.0,5.0,3674.0,14845.0,2019.0,33.590623,-117.862080,10777000.0
...,...,...,...,...,...,...,...,...,...
12446,180000.0,3.0,3.0,1614.0,7841.0,1991.0,38.952577,-122.739291,370600.0
10235,180000.0,2.0,1.0,768.0,7405.0,1955.0,33.824583,-116.391381,433000.0
5406,180000.0,4.0,3.0,2550.0,308840.4,1979.0,39.298874,-122.489578,173600.0
19163,179000.0,2.0,1.0,740.0,8712.0,1961.0,38.861356,-122.714956,409000.0


In [38]:
# linear regression
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score


# model = smf.ols(formula = "ClosePrice ~  YearBuilt + BathroomsTotalInteger + BedroomsTotal + NewConstructionYN + LotSizeSquareFeet + Latitude + Longitude + SpatialLag_Price", data = df_filtered_train)

model = smf.ols(formula = "ClosePrice ~ BedroomsTotal + BathroomsTotalInteger + LivingArea + LotSizeSquareFeet + YearBuilt + Latitude + Longitude + SpatialLag_Price", data = df_filtered_train).fit()
print(model.summary())

# print(df_filtered_train.columns)
# print(df_test_cleaned.columns)

# df_filtered_train
predictions = model.predict(df_test_cleaned)
actual = df_test_cleaned["ClosePrice"]
# predictions = model.predict(df_test_cleaned[["PropertySubType", "YearBuilt", "BathroomsTotalInteger", "BedroomsTotal", "NewConstructionYN", "LotSizeSquareFeet"]])
# actual = df_test_cleaned["ClosePrice"]
r_squared = r2_score(actual, predictions)
mape_lin = mape(actual, predictions)
print("R_squared for linear regression: ", r_squared)
print("MaPe for linear regression: ", mape_lin)

                            OLS Regression Results                            
Dep. Variable:             ClosePrice   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     131.5
Date:                Sat, 21 Feb 2026   Prob (F-statistic):          4.54e-220
Time:                        13:56:00   Log-Likelihood:            -1.1918e+06
No. Observations:               68384   AIC:                         2.384e+06
Df Residuals:                   68375   BIC:                         2.384e+06
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept              2.853e+

In [39]:
#random forest Rsquared
import random
from sklearn.ensemble import RandomForestRegressor
rf =  RandomForestRegressor(n_estimators=2000, max_depth=None, min_samples_leaf=500, n_jobs = -1, random_state=42)
x_train = df_filtered_train[["BedroomsTotal","BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude", "SpatialLag_Price"]]
print(type(x_train))
y_train = df_filtered_train["ClosePrice"]
rf.fit(x_train, y_train)
x_test = df_test_cleaned[["BedroomsTotal","BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude", "SpatialLag_Price"]]
y_pred = rf.predict(x_test)
r2 = r2_score(actual, y_pred)
mape_rf = mape(actual, y_pred)
print("Rsquared for random forest: ", r2)
print("MaPe for random forest: ", mape_rf)


<class 'pandas.core.frame.DataFrame'>
Rsquared for random forest:  0.5765718508115991
MaPe for random forest:  0.2981593164585019


In [40]:
#Gradient Boosting R-Squared
from xgboost import XGBRegressor
y_test = df_test_cleaned["ClosePrice"]
gb_model = XGBRegressor(n_estimators=300,
    learning_rate=0.001,
    max_depth=None,
    subsample = 0.5,
    reg_lambda=10.0,
    objective="reg:squarederror",
    random_state=42)
gb_model.fit(x_train, y_train)

predictions = gb_model.predict(x_test)
r2_gb = r2_score(actual, predictions)
mape_gb = mape(actual, predictions)
print("Rsquared gradient boosting: ", r2_gb)
print("MaPe gradient boosting: ", mape_gb)

Rsquared gradient boosting:  0.28090683711972786
MaPe gradient boosting:  0.7227094600244611
